# DateVision model server (Google Colab)

This notebook loads your trained **MobileNetV3-Large** grading model (`best_model.pth` and `class_names.json`) and serves it as a small web API. The DateVision web app sends photos to this API and shows the result.

**How to use it**
1. Set **Runtime > Change runtime type > GPU** (a T4 is enough).
2. Create a free account at https://ngrok.com and copy your **authtoken** from the dashboard. In Colab, click the **key icon** (Secrets) in the left sidebar, add a secret named `NGROK_TOKEN`, paste the token as its value and switch on **Notebook access**. Never paste the token into a chat or share the notebook with it inside.
3. Run all cells from top to bottom.
4. Copy the public URL printed by the *Start the server* cell.
5. In DateVision, click the status button in the top bar, turn **Demo mode** off, paste the URL, press **Test connection**, then **Save**.

Keep this tab open while you use the app. When the Colab session stops, or you run the server cell again, the URL changes and must be pasted into the app again.

**What the app expects from this server**

| Route | Purpose |
|---|---|
| `GET /health` | Connection test. Returns the model name and device. |
| `POST /predict` | Receives a photo (form field `file`). Returns the probability for each of the four grades, plus an optional Grad-CAM heatmap and the gatekeeper verdict. |
| `GET /metrics` | Optional. Returns your evaluation results for the Performance section. |

The **confidence threshold** is applied inside the web app (the slider above the workspace), so you can change it at any time without restarting this server.

In [ ]:
!pip -q install fastapi uvicorn python-multipart pyngrok nest_asyncio

## 1. Settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- Edit these if your files are somewhere else ----
MODEL_PATH   = "/content/drive/MyDrive/Mozawati_Outputs/best_model.pth"
META_PATH    = "/content/drive/MyDrive/Mozawati_Outputs/class_names.json"
METRICS_PATH = "/content/drive/MyDrive/Mozawati_Outputs/metrics.json"   # optional, written by the evaluation cell

# ---- ngrok token ----
# Best way: click the key icon in Colab's left sidebar (Secrets), add a secret named NGROK_TOKEN,
# paste your authtoken as the value and switch on "Notebook access". Then the token is never stored in this notebook.
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_TOKEN")
except Exception:
    NGROK_TOKEN = "PASTE_YOUR_NGROK_AUTHTOKEN_HERE"   # fallback: paste it here, but never share the notebook with it inside


## 2. Load the grading model and the gatekeeper

* The **grading model** uses exactly the architecture you trained: MobileNetV3-Large with `Dropout(0.4)` and a 4-way output layer. The class order comes from `class_names.json`, so it always matches your training labels.
* The **Smart Gatekeeper** is a standard ImageNet MobileNetV3-Large. It looks at the top-5 ImageNet guesses for the photo and marks the photo as **not a date** when:
  * it sees a person, animal or vehicle and nothing on the allowed list (fruit, food, plants, containers, and things a premium date can look like, such as chocolate, snail, stone or bean), or
  * it is very sure the photo shows another fruit or food, for example an apple or a banana (`OTHER_FRUIT_KEYWORDS`).

  The web app then shows **Not confident** with no grade and no percentage. Set `STRICT_MODE = True` if you also want to block every photo that has no food, fruit or plant word at all. You can switch the gatekeeper off in the app settings.


In [ ]:
import io, os, time, json, base64
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
from matplotlib import cm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(META_PATH, "r") as f:
    meta = json.load(f)
CLASS_NAMES = meta["classes"]
assert len(CLASS_NAMES) == 4, f"The app expects exactly 4 grades, but class_names.json has {len(CLASS_NAMES)}."
print("Classes (in training order):", CLASS_NAMES)

# --- Grading model (same architecture as in training) ---
model = models.mobilenet_v3_large(weights=None)
in_features = model.classifier[3].in_features
model.classifier[2] = nn.Dropout(p=0.4)
model.classifier[3] = nn.Linear(in_features, len(CLASS_NAMES))
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE).eval()

# Must match the preprocessing used in training
preprocess = transforms.Compose([
    transforms.Resize((meta["img_size"], meta["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=meta["mean"], std=meta["std"]),
])

# --- Smart Gatekeeper (downloads the ImageNet weights the first time) ---
gate_weights = models.MobileNet_V3_Large_Weights.DEFAULT
gate_model = models.mobilenet_v3_large(weights=gate_weights).to(DEVICE).eval()
GATE_CATEGORIES = gate_weights.meta["categories"]

# Blacklist: humans, animals and vehicles that must be blocked.
BLOCKED_KEYWORDS = [
    "dog", "cat", "person", "man", "woman", "boy", "girl", "human",
    "bird", "fish", "monkey", "bear", "horse", "cow", "sheep", "lion",
    "tiger", "car", "truck", "vehicle", "bicycle", "motorcycle",
]

# Whitelist: food, plants and containers, plus what premium dates can look like to the AI
# (chocolate, snail, bean and similar).
ALLOWED_KEYWORDS = [
    "fruit", "food", "produce", "fig", "apple", "orange", "banana", "lemon",
    "pomegranate", "acorn", "chestnut", "grocery", "basket", "bowl", "plate",
    "tray", "dish", "market", "plant", "leaf", "strawberry", "pineapple",
    "chocolate", "sauce", "snail", "slug", "stone", "rock", "bean", "coffee", "dough", "sweet",
]
# Other fruits and foods that are clearly NOT dates. They block a photo only when ImageNet is very sure
# (its top guess is at or above OTHER_FRUIT_MIN_CONF), because shiny or unripe dates can resemble a few fruits.
# 'granny smith' is how ImageNet names a green apple. Add or remove words after testing your own photos.
OTHER_FRUIT_KEYWORDS = [
    "granny smith", "banana", "orange", "lemon", "pineapple", "jackfruit", "custard apple",
    "cucumber", "zucchini", "broccoli", "cauliflower", "head cabbage", "mushroom",
]
OTHER_FRUIT_MIN_CONF = 0.60

# STRICT_MODE = False: block people, animals, vehicles and the other fruits above (your Smart Gatekeeper).
# STRICT_MODE = True : also block every photo that has no food, fruit or plant word in its top-5 guesses.
STRICT_MODE = False

print("Models loaded on", DEVICE)

## 3. Prediction, gatekeeper and Grad-CAM

In [ ]:
def to_tensor(img):
    return preprocess(img).unsqueeze(0).to(DEVICE)

def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def gate_decision(labels, top1_conf):
    """Returns (passed, reason). `labels` are the ImageNet top-5 names in lower case, best first."""
    blocked_object = any(b in lab for lab in labels for b in BLOCKED_KEYWORDS)
    safe_context = any(k in lab for lab in labels for k in ALLOWED_KEYWORDS)
    other_fruit = top1_conf >= OTHER_FRUIT_MIN_CONF and any(k in labels[0] for k in OTHER_FRUIT_KEYWORDS)
    if other_fruit:
        return False, "other_fruit"
    if blocked_object and not safe_context:
        return False, "blocked_object"
    if STRICT_MODE and not safe_context:
        return False, "not_date_like"
    return True, None

def run_gatekeeper(x):
    with torch.no_grad():
        probs = F.softmax(gate_model(x), dim=1)[0]
        top_p, top_i = torch.topk(probs, 5)
    labels = [GATE_CATEGORIES[i.item()].lower() for i in top_i]
    passed, reason = gate_decision(labels, float(top_p[0]))
    return {
        "enabled": True,
        "passed": bool(passed),
        "reason": reason,
        "top": [{"label": l, "confidence": float(p)} for l, p in zip(labels, top_p)],
    }

def predict_fast(x):
    sync(); t0 = time.perf_counter()
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)[0]
    sync()
    return probs.cpu().numpy(), (time.perf_counter() - t0) * 1000

def predict_with_cam(x, out_size):
    """Prediction plus a Grad-CAM heatmap for the predicted class."""
    layer = model.features[-1]
    store = {}

    def fwd_hook(module, inp, out):
        store["act"] = out
        out.register_hook(lambda g: store.__setitem__("grad", g))

    handle = layer.register_forward_hook(fwd_hook)
    try:
        sync(); t0 = time.perf_counter()
        model.zero_grad()
        with torch.enable_grad():
            logits = model(x)
            probs = F.softmax(logits, dim=1)[0]
            logits[0, int(probs.argmax())].backward()
        sync()
        ms = (time.perf_counter() - t0) * 1000
    finally:
        handle.remove()

    w = store["grad"].mean(dim=(2, 3), keepdim=True)
    cam_map = torch.relu((w * store["act"]).sum(1, keepdim=True))
    cam_map = F.interpolate(cam_map, size=(meta["img_size"], meta["img_size"]), mode="bilinear", align_corners=False)[0, 0]
    cam_map = ((cam_map - cam_map.min()) / (cam_map.max() - cam_map.min() + 1e-8)).detach().cpu().numpy()
    heat = (cm.jet(cam_map)[..., :3] * cam_map[..., None] * 255).astype(np.uint8)
    heat = Image.fromarray(heat).resize(out_size)          # same size as the uploaded photo
    buf = io.BytesIO(); heat.save(buf, format="PNG")
    return probs.detach().cpu().numpy(), ms, base64.b64encode(buf.getvalue()).decode()

### Optional: check what the gatekeeper thinks of any photo

Run this cell and upload a date photo, an apple, a banana or anything else. It shows whether the photo passes or is blocked, plus ImageNet's top-5 guesses. Use it to check that your real dates pass and other fruits are blocked. If a real date is blocked, remove the matching word from `OTHER_FRUIT_KEYWORDS`; if an apple passes, add the word you see in its top-5 list. After changing a list, run the model-loading cell again.

In [ ]:
# See what the Smart Gatekeeper thinks of any photo (a date, an apple, a banana ...).
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    img = Image.open(io.BytesIO(data)).convert("RGB")
    g = run_gatekeeper(to_tensor(img))
    verdict = "PASSES" if g["passed"] else f"BLOCKED ({g['reason']})"
    print(f"\n{name}: {verdict}")
    for t in g["top"]:
        print(f"   {t['label']:<32} {t['confidence'] * 100:5.1f}%")

## 4. The API the web app talks to

In [ ]:
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="DateVision API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/health")
def health():
    return {"status": "ok", "model": "mobilenet_v3_large", "device": str(DEVICE), "classes": CLASS_NAMES}

@app.post("/predict")
async def predict(file: UploadFile = File(...), cam: int = 1, gate: int = 1):
    try:
        img = Image.open(io.BytesIO(await file.read())).convert("RGB")
    except Exception:
        raise HTTPException(status_code=400, detail="The uploaded file is not a valid image.")

    x = to_tensor(img)
    gate_info = run_gatekeeper(x) if gate else None
    rejected = bool(gate_info) and not gate_info["passed"]

    if cam and not rejected:
        probs, ms, heat = predict_with_cam(x, img.size)
    else:
        probs, ms = predict_fast(x)
        heat = None

    out = {
        "model": "mobilenet_v3_large",
        "inference_ms": round(ms, 1),
        "predictions": [{"label": n, "confidence": float(p)} for n, p in zip(CLASS_NAMES, probs)],
    }
    if gate_info:
        out["gatekeeper"] = gate_info
    if heat:
        out["gradcam"] = heat
    return out

@app.get("/metrics")
def metrics():
    if not os.path.exists(METRICS_PATH):
        raise HTTPException(status_code=404, detail="metrics.json not found")
    with open(METRICS_PATH) as f:
        return json.load(f)

## 5. Start the server and get the public URL

In [ ]:
import os, threading, time, requests, uvicorn, nest_asyncio
from pyngrok import ngrok, conf
from pyngrok.exception import PyngrokNgrokError

assert NGROK_TOKEN and "PASTE" not in NGROK_TOKEN, (
    "No ngrok token found. In Colab, click the key icon in the left sidebar, add a secret named NGROK_TOKEN with your token as the value, and switch on Notebook access. Then run the Settings cell again. (Or paste the token in the Settings cell.)"
)

nest_asyncio.apply()

# If you ran this cell before, stop that server first. Otherwise port 8000 stays busy and the old code keeps running.
if "server" in globals():
    server.should_exit = True
    time.sleep(2)

ngrok.kill()
try:
    ngrok.set_auth_token(NGROK_TOKEN)
except PyngrokNgrokError as e:
    if "already set" not in str(e):
        raise
    # An older token is saved on this Colab machine and ngrok refuses to overwrite it.
    # Remove that file so the token you entered above is the one that gets used.
    cfg = getattr(conf, "DEFAULT_NGROK_CONFIG_PATH", os.path.expanduser("~/.config/ngrok/ngrok.yml"))
    if os.path.exists(cfg):
        os.remove(cfg)
    ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000)

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()

# Wait until the server really answers before showing the URL
for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("The server did not start. Run this cell again. If it still fails, use Runtime > Restart session and run all cells from the top.")

print("Server is running.")
print("Paste this URL into DateVision > Model server URL:")
print(tunnel.public_url)


## 6. Optional: test the server from Colab

Use this if the web app cannot connect and you want to know whether the problem is the server or the app. Put one date photo (.jpg or .png) in the same Drive folder as your model, and this cell will find it by itself. You can also set `TEST_IMAGE` to an exact path.

In [ ]:
import glob, requests

if "tunnel" not in globals():
    raise RuntimeError("Run the 'Start the server' cell first. It creates the public URL.")

# Option 1: leave TEST_IMAGE empty to use the first .jpg/.jpeg/.png found next to your model.
# Option 2: set TEST_IMAGE to the exact path of a date photo.
TEST_IMAGE = ""

if not TEST_IMAGE:
    folder = os.path.dirname(MODEL_PATH)
    found = []
    for pattern in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG"):
        found += glob.glob(os.path.join(folder, pattern))
    TEST_IMAGE = sorted(found)[0] if found else ""

if not TEST_IMAGE:
    print(f"No test photo found in {os.path.dirname(MODEL_PATH)}.")
    print("This test is optional. To run it, put one date photo (.jpg or .png) in that folder, or set TEST_IMAGE above, then run this cell again.")
else:
    print("Using photo:", TEST_IMAGE)
    headers = {"ngrok-skip-browser-warning": "1"}
    health = requests.get(tunnel.public_url + "/health", headers=headers)
    print("Health:", health.status_code, health.text[:200])

    with open(TEST_IMAGE, "rb") as f:
        r = requests.post(tunnel.public_url + "/predict", files={"file": f}, headers=headers)
    print("Status:", r.status_code)
    try:
        result = r.json()
    except ValueError:
        print("The server did not return JSON. It said:", r.text[:300])
        raise
    result.pop("gradcam", None)          # the heatmap is long, so hide it here
    print(json.dumps(result, indent=2))


## 7. Optional: send your real evaluation results to the app

This section runs your model on your **test images**, measures accuracy, precision, recall, F1-score and speed, and saves them to `metrics.json`. The web app then fills its **Performance** section with these real numbers (reload the app page afterwards).

1. Run the first cell below. It searches your Drive, shared drives and Colab's local disk, and lists the folders that look like a dataset. If it says Drive is not mounted, run the Settings cell first.
2. Copy the path of your **test** folder (about 10% of your images) into `TEST_DIR` in the second cell, then run it.

The test folder must contain one sub-folder per grade, for example:

```
test/
  Premium/
  Semi-Ripe/
  Ungraded/
  Unripe/
```

Folder names are matched to the classes in `class_names.json` automatically, so the order cannot get mixed up.

In [ ]:
# Finds your dataset for you. It checks Google Drive, shared drives and Colab's local disk for folders that
# look like a dataset (one sub-folder of photos per grade). This can take a minute because Drive is slow.
import os

def _norm(s):
    return s.lower().replace("-", "").replace("_", "").replace(" ", "")

GRADE_KEYS = ["premium", "semiripe", "ungraded", "unripe"]
IMAGE_EXT = (".jpg", ".jpeg", ".png")
ROOTS = ["/content/drive/MyDrive", "/content/drive/Shareddrives", "/content"]
SKIP_PREFIXES = ("/content/drive", "/content/sample_data", "/content/.config")

if not os.path.isdir("/content/drive/MyDrive"):
    print("WARNING: Google Drive is NOT mounted in this session.")
    print("Run the Settings cell (drive.mount) first, then run this cell again.\n")

direct_images = {}     # folder -> number of photos directly inside it
zip_files = []
for root in ROOTS:
    if not os.path.isdir(root):
        continue
    for dirpath, dirnames, files in os.walk(root):
        if root == "/content" and dirpath.startswith(SKIP_PREFIXES):
            dirnames[:] = []
            continue
        if dirpath[len(root):].count(os.sep) >= 6:        # do not search deeper than 6 levels
            dirnames[:] = []
            continue
        n = sum(1 for f in files if f.lower().endswith(IMAGE_EXT))
        if n:
            direct_images[dirpath] = n
        zip_files += [os.path.join(dirpath, f) for f in files if f.lower().endswith(".zip")]

parents = {}
for folder, n in direct_images.items():
    parents.setdefault(os.path.dirname(folder), []).append((os.path.basename(folder), n))
datasets_found = [(par, subs) for par, subs in parents.items() if len(subs) >= 3]
datasets_found.sort(key=lambda r: -sum(n for _, n in r[1]))

if datasets_found:
    print("Folders that look like a dataset (one sub-folder of photos per grade):\n")
    for par, subs in datasets_found:
        total = sum(n for _, n in subs)
        match = all(any(k in _norm(s) for s, _ in subs) for k in GRADE_KEYS)
        print(f"{total:>6} photos  {par}" + ("   <-- grade names match" if match else ""))
        print("          " + ", ".join(f"{s} ({n})" for s, n in sorted(subs)))
    print("\nYour test set holds about 10% of the data (around 400 photos for a 4,000 photo dataset).")
    print("Copy the path of that folder into TEST_DIR in the next cell.")
else:
    print("No dataset-style folder was found in Drive or on Colab's local disk.")
    if os.path.isdir("/content/drive/MyDrive"):
        print("Top level of your Drive:", sorted(os.listdir("/content/drive/MyDrive"))[:40])

if zip_files:
    print("\nZip files found (your dataset may still be zipped, so unzip it first):")
    for z in zip_files:
        print(f"  {os.path.getsize(z) / 1e6:>7.0f} MB  {z}")


**No test folder found?** If you split the data inside your training code (for example with `random_split`), the test images only existed in memory. Either save them into a folder with one sub-folder per grade from your training notebook, or recreate the same split with the same random seed you used in training. Do not use training or validation images here: the numbers would be too high and would not be a fair test.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from torchvision import datasets
from torch.utils.data import DataLoader

# Folder with your TEST images: one sub-folder per grade, the same layout as your training data.
TEST_DIR = "/content/drive/MyDrive/Mozawati_Dataset/test"      # <-- change this to your test folder

assert os.path.isdir(TEST_DIR), (
    f"Test folder not found: {TEST_DIR}\n"
    "Set TEST_DIR to the folder that contains one sub-folder per grade (for example Premium, Semi-Ripe, Ungraded, Unripe)."
)

def _norm(s):
    return s.lower().replace("-", "").replace("_", "").replace(" ", "")

test_ds = datasets.ImageFolder(TEST_DIR, transform=preprocess)
print("Test folders found:", test_ds.classes, "|", len(test_ds), "images")

# Match every folder name to the right class in class_names.json, so the order can never get mixed up.
folder_to_idx = {}
for fi, fname in enumerate(test_ds.classes):
    match = [i for i, n in enumerate(CLASS_NAMES) if _norm(fname) in _norm(n) or _norm(n) in _norm(fname)]
    assert len(match) == 1, f"Cannot match the test folder '{fname}' to exactly one of {CLASS_NAMES}"
    folder_to_idx[fi] = match[0]

# 1) Predictions on the whole test set
loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)
y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in loader:
        y_pred += model(xb.to(DEVICE)).argmax(1).cpu().tolist()
        y_true += [folder_to_idx[int(y)] for y in yb]

# 2) Speed: one image at a time, model only (warm-up first)
one = test_ds[0][0].unsqueeze(0).to(DEVICE)
with torch.no_grad():
    for _ in range(5):
        model(one)
    sync(); t0 = time.perf_counter()
    for _ in range(50):
        model(one)
    sync()
fps = 50 / (time.perf_counter() - t0)

# 3) Metrics. The app lists the grades in this order, so the confusion matrix is re-ordered to match.
APP_ORDER = ["Unripe", "Semi-ripe", "Premium", "Ungraded"]
perm = [next(i for i, n in enumerate(CLASS_NAMES) if _norm(k) in _norm(n)) for k in APP_ORDER]

p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
cmx = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))[np.ix_(perm, perm)]

metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_precision": float(p),
    "macro_recall": float(r),
    "macro_f1": float(f1),
    "fps": float(fps),
    "confusion_matrix": cmx.tolist(),
}
with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
print(f"Accuracy {metrics['accuracy']:.4f} | Macro F1 {metrics['macro_f1']:.4f} | {fps:.1f} FPS")
print("Saved to", METRICS_PATH)
